In [ ]:
%load_ext autoreload

import os
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import numpy as np

import mcfs
import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget
# %matplotlib inline

### Plotting utils

In [ ]:
linestyles = ["-", "--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 1))]
cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in range(cmap.N)]

def _subset_to_text(subset):
    """
    Convert a subset label tuple into a compact readable string.
    Example:
        ('HI-LyA', 'SiIII-1206') -> 'HI-LyA·SiIII-1206'
    """
    subset = tuple(subset)
    if len(subset) == 1:
        return subset[0]
    return "·".join(subset)

def _canonical_sym_pair(A, B):
    """
    Canonical representative of the symmetric pair (A,B) ~ (B,A).
    """
    return min((A, B), (B, A))

# Load fake_spectra lines

In [ ]:
base_dir = "/home/STORAGE"
snap_num = 33

sim_name = "TNG50-2"
delta_grid = 0.5 # 0.2
res_kms = 3.0 # 2.0

# sim_name = "TNG100-1"
# delta_grid = 0.5
# res_kms = 2.0

data = mcfs.load_runs.load_data(base_dir=base_dir, sim_name=sim_name, snap_num=snap_num, delta_grid=delta_grid, res_kms=res_kms, preset="lya_si")

In [ ]:
tau_loaded = data["tau"]
v_kms = data["v_kms"]

print(data["field_keys"]["tau"])
print(tau_loaded.shape)

### postproc tau

In [ ]:
_tau_factor = 1e11
array_tau_factors = np.array([1.0, _tau_factor, None, None], dtype=object) # One entry per line in tau_loaded. Use None for lines you want to discard

keep_mask = np.array([f is not None for f in array_tau_factors], dtype=bool) # Keep only entries with a valid scaling factor
tau_factors_kept = np.array(array_tau_factors[keep_mask], dtype=float) # Convert kept factors to float
tau_scaled = tau_loaded[keep_mask] * tau_factors_kept[:, None, None] # Filter tau_loaded and apply the corresponding scaling

In [ ]:
print(tau_scaled.shape)
print((tau_scaled.shape[1]/3)**0.5)

# overflux

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA") # "SiII-1190", "SiII-1193"

In [ ]:
tau_cube = mcfs.overflux_tools.OpticalDepthCube(tau=tau_scaled, line_bundle=line_bundle)
tau_shifted = tau_cube.periodic_shift(x=v_kms, period=np.max(v_kms))
overflux, overflux_total = tau_cube.build_overflux(tau=tau_shifted, ens_axis=0)
overflux_subsets, subset_labels = tau_cube.build_subset_fields(overflux=overflux)
C_means = np.mean(np.mean(overflux_subsets, axis=-1), axis=-1)

In [ ]:
# tau_scaled shape: (N_lines, N_skewers, N_spectral)
print(f"tau shape: {tau_cube.shape}")
print(f"line IDs: {tau_cube.line_ids}")
print(f"species keys: {tau_cube.species_keys}")
print(f"reference line: {tau_cube.ref_line_id}")
print(f"delta_v array: {tau_cube.delta_v}")
print(f"delta_v dict: {tau_cube.delta_v_dict}")
print(f"tau_shifted shape: {tau_shifted.shape}")
print(f"overflux shape: {overflux.shape}")
print(f"overflux_total shape: {overflux_total.shape}")
print(f"overflux_subsets shape: {overflux_subsets.shape}")
print(f"subset_labels: {subset_labels}")
print(f"means of subsets: {C_means}")

# Plots

In [ ]:
# Configuration
n_plot = 2               # number of random skewers to display
seed = 137              # for reproducibility
tau_plot = tau_shifted  # Use tau_cube.tau if you want the unshifted optical depth, Use tau_shifted if you want the shifted optical depth

rng = np.random.default_rng(seed) # Random skewer selection
ii_list = np.sort(rng.choice(tau_cube.n_skewers, size=min(n_plot, tau_cube.n_skewers), replace=False))

xlabel = r"$c\, \ln\left( \frac{\lambda}{\lambda_\dagger}\right)\; \left[\mathrm{km\,/\,s}\right]$"

# Shared metadata / styling
line_ids = tau_cube.line_ids
n_lines = tau_cube.n_lines

cmap = plt.get_cmap("tab10")
colors = {line_id: cmap(i % cmap.N) for i, line_id in enumerate(line_ids)}

line_handles = [
    Line2D([0], [0], color=colors[line_id], lw=3, label=line_id) for line_id in line_ids
]
total_handle = Line2D([0], [0], color="k", lw=3, ls=":", alpha=0.7, label="Total")

# Combined plotting
for ii in ii_list:
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    tau_tot = np.sum(tau_plot[:, ii, :], axis=0)
    F_lines = np.exp(-tau_plot[:, ii, :])
    F_tot = np.prod(F_lines, axis=0)

    for jj, line_id in enumerate(line_ids):
        color = colors[line_id]
        
        axes[0].plot(v_kms, tau_plot[jj, ii], color=color, lw=1.0, alpha=0.7) # Panel 1: optical depth
        axes[1].plot(v_kms, F_lines[jj], color=color, lw=1.0, alpha=0.7) # Panel 2: transmitted flux per line
        axes[2].plot(v_kms, overflux[jj, ii], color=color, lw=1.0, alpha=0.7) # Panel 3: per-line overflux

    # Total curves
    axes[0].plot(v_kms, tau_tot, color="k", lw=2.0, ls=":", alpha=0.7)
    axes[1].plot(v_kms, F_tot,   color="k", lw=2.0, ls=":", alpha=0.7)
    axes[2].plot(v_kms, overflux_total[ii], color="k", lw=2.0, ls=":", alpha=0.7)
    axes[2].axhline(0.0, color="k", lw=1.0, alpha=0.6)

    # Labels / scales
    axes[0].set_ylabel(r"$\tau$")
    axes[0].set_yscale("log")

    axes[1].set_ylabel(r"$e^{-\tau}$")

    axes[2].set_ylabel(r"$\delta$")
    axes[2].set_xlabel(xlabel)

    # Titles
    axes[0].set_title(f"Skewer index: {ii}")

    # Legend only once, in top panel
    axes[0].legend(handles=line_handles + [total_handle], loc="upper right", frameon=True, fontsize=16)

    plt.tight_layout()
    plt.show()

# P1D

In [ ]:
P1D = mcfs.P1D.P1DAnalyzer(lambda_or_v=v_kms, optical_depth_cube=tau_cube) # Initialize the P1D analyzer on the same spectral grid
omega_tot, P1D_tot = P1D.compute_total_P1D(overflux_total=overflux_total, axis=-1, subtract_mean=False, drop_zero_mode=True) # Total P1D
P1D_tot_mean = np.mean(P1D_tot, axis=0)

omega, P1D_catalog, by_order = P1D.compute_subset_P1D_catalog(
    subset_fields=overflux_subsets, subset_labels=subset_labels, axis=-1, subtract_mean=False, drop_zero_mode=True, # Full subset-pair catalog
)
omega, avg_P1D_catalog = P1D.compute_average_P1D_catalog(P1D_catalog=P1D_catalog, average_axis=0, return_std=False, return_sem=True, symmetrize=True) # Average over skewers

In [ ]:
print(f"omega_tot shape: {omega_tot.shape}")
print(f"P1D_tot shape: {P1D_tot.shape}")
print(f"P1D_tot_mean shape: {P1D_tot_mean.shape}")

print(f"number of subset labels: {len(subset_labels)}")
print(f"first few subset labels: {subset_labels[:5]}")

print(f"number of P1D catalog entries: {len(P1D_catalog)}")
for key, ii in enumerate(P1D_catalog.keys()):
    print(f"entry {key}: {ii}")
print(f"available total orders: {sorted(by_order.keys())}")

example_key = next(iter(avg_P1D_catalog))
print(f"example averaged key: {example_key}")
print(f"example label: {avg_P1D_catalog[example_key]['label']}")
print(f"example mean shape: {avg_P1D_catalog[example_key]['P1D_mean'].shape}")
if 'P1D_sem' in avg_P1D_catalog[example_key]:
    print(f"example sem shape: {avg_P1D_catalog[example_key]['P1D_sem'].shape}")

In [ ]:
cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in range(cmap.N)]

# Group terms by symmetric pair, but keep both members
grouped_terms = {}
for (A, B), entry in avg_P1D_catalog.items():
    canon = _canonical_sym_pair(A, B)
    grouped_terms.setdefault(canon, []).append(((A, B), entry))

# Sort groups by total order, then lexicographically by canonical pair
sorted_groups = sorted(grouped_terms.items(), key=lambda item: (item[1][0][1]["total_order"], item[0][0], item[0][1]))

orders = sorted({group[0][1]["total_order"] for _, group in sorted_groups})
order_to_ls = {order: linestyles[i % len(linestyles)] for i, order in enumerate(orders)}

fig, ax = plt.subplots(figsize=(12, 7))

term_handles = []
term_labels = []

sum_P1D_terms = np.zeros_like(omega)

for i, (canon, group) in enumerate(sorted_groups):
    color = colors[i % len(colors)]

    # Sort members inside the symmetric group so (A,B) and (B,A) appear together
    group = sorted(group, key=lambda x: (x[0][0], x[0][1]))

    for j, ((A, B), entry) in enumerate(group):
        print(i, j, A, B, entry["total_order"])

        mean = np.real(entry["P1D_mean"])
        sem = np.asarray(entry.get("P1D_sem", np.zeros_like(mean)))

        order = entry["total_order"]
        ls = order_to_ls[order]

        A_lab = _subset_to_text(A)
        B_lab = _subset_to_text(B)
        label = f"P[{A_lab}|{B_lab}]"

        sum_P1D_terms += mean

        # Same color/style for the symmetric pair; small alpha tweak helps if both differ
        line_alpha = 0.95 if j == 0 else 0.65
        band_alpha = 0.15 if j == 0 else 0.08

        ax.plot(omega, mean, lw=2, ls=ls, color=color, alpha=line_alpha)
        ax.fill_between(omega, mean - sem, mean + sem, color=color, alpha=band_alpha)

        term_handles.append(Line2D([0], [0], color=color, lw=2, ls=ls, alpha=line_alpha))
        term_labels.append(label)

# total spectrum
ax.plot(omega_tot, np.real(P1D_tot_mean), lw=1.2, ls="-", color="k", label="total")

P1D_recovered = sum_P1D_terms / (1 + C_means[-1])**2
ax.plot(omega_tot, P1D_recovered, lw=2.2, ls="--", color="gray", label="sum P1D terms")

ax.set_xscale("log")
ax.set_yscale("symlog")
ax.set_xlabel(r"$\omega$")
ax.set_ylabel("P1D")

# Legend for terms
leg1 = ax.legend(term_handles, term_labels, loc="upper right", fontsize=12, ncol=1, frameon=True)
ax.add_artist(leg1)

# Legend for perturbative / total order
order_handles = [
    Line2D([0], [0], color="k", lw=2, ls=order_to_ls[o], label=rf"$\mathcal{{O}}({o})$")
    for o in orders
]
ax.legend(handles=order_handles, loc="lower left", fontsize=16, frameon=True, ncol=len(order_handles))

plt.tight_layout()
plt.show()

In [ ]:
max_x = 0.045
y_min = 7e-3
y_max = 6e-1

xlabel = r"$k_\parallel \left[\mathrm{km}^{-1}\,\mathrm{s}\right]$"
ylabel = r"$k_\parallel P_{\mathrm{1D}} / \pi$"

# Group terms by symmetric pair, but keep both members
grouped_terms = {}
for (A, B), entry in avg_P1D_catalog.items():
    canon = _canonical_sym_pair(A, B)
    grouped_terms.setdefault(canon, []).append(((A, B), entry))

# Sort groups by total order, then lexicographically by canonical pair
sorted_groups = sorted(
    grouped_terms.items(),
    key=lambda item: (item[1][0][1]["total_order"], item[0][0], item[0][1])
)

orders = sorted({group[0][1]["total_order"] for _, group in sorted_groups})
order_to_ls = {order: linestyles[i % len(linestyles)] for i, order in enumerate(orders)}

fig, ax = plt.subplots(figsize=(12, 7))

term_handles = []
term_labels = []

omega = np.asarray(omega, dtype=float)
omega_tot = np.asarray(omega_tot, dtype=float)

mask_terms = (omega >= 0.0) & (omega <= max_x)
mask_total = (omega_tot >= 0.0) & (omega_tot <= max_x)

for i, (canon, group) in enumerate(sorted_groups):
    color = colors[i % len(colors)]

    # Sort members inside each symmetric pair so they appear together
    group = sorted(group, key=lambda x: (x[0][0], x[0][1]))

    for j, ((A, B), entry) in enumerate(group):
        mean = np.real(np.asarray(entry["P1D_mean"]))
        sem  = np.asarray(entry.get("P1D_sem", np.zeros_like(mean)))

        # Transform to k P1D / pi
        y = omega * mean / np.pi
        y_sem = omega * sem / np.pi

        order = entry["total_order"]
        ls = order_to_ls[order]

        A_lab = _subset_to_text(A)
        B_lab = _subset_to_text(B)
        label = f"P[{A_lab}|{B_lab}]"

        # Same color for symmetric pair; slight alpha change helps distinguish them
        line_alpha = 0.95 if j == 0 else 0.65
        band_alpha = 0.15 if j == 0 else 0.08

        ax.plot(omega[mask_terms], y[mask_terms], lw=2, ls=ls, color=color, alpha=line_alpha)
        ax.fill_between(
            omega[mask_terms],
            np.clip(y[mask_terms] - y_sem[mask_terms], 1e-300, None),
            np.clip(y[mask_terms] + y_sem[mask_terms], 1e-300, None),
            color=color, alpha=band_alpha,
        )

        term_handles.append(Line2D([0], [0], color=color, lw=2, ls=ls, alpha=line_alpha))
        term_labels.append(label)

# Total spectrum
y_tot = omega_tot * np.real(np.asarray(P1D_tot_mean)) / np.pi
ax.plot(omega_tot[mask_total], y_tot[mask_total], lw=1.4, ls="-", color="k", label="total")

# Axes styling
ax.set_xlim(0.0, max_x)
# ax.set_ylim(y_min, y_max)
# ax.set_yscale("log")
ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)

# Legend for terms
leg1 = ax.legend(term_handles, term_labels, loc="upper right", fontsize=14, ncol=1, frameon=True)
ax.add_artist(leg1)

# Legend for perturbative / total order
order_handles = [Line2D([0], [0], color="k", lw=2, ls=order_to_ls[o], label=rf"$\mathcal{{O}}({o})$") for o in orders]
order_handles.append(Line2D([0], [0], color="k", lw=1.4, ls="-", label="total"))

ax.legend(handles=order_handles, loc="lower left", fontsize=14, frameon=True, ncol=len(order_handles))

plt.tight_layout()
plt.show()

In [ ]:
# Configuration
alpha_line_id = "HI-LyA"   # reference "alpha" line
max_x = 0.025
y_min = None
y_max = None

xlabel = r"$k_\parallel \left[\mathrm{km}^{-1}\,\mathrm{s}\right]$"
ylabel = rf"$P_{{\mathrm{{total}}}} / P_{{{alpha_line_id}}}$"

# Extract the alpha auto-spectrum term
alpha_key = ((alpha_line_id,), (alpha_line_id,))
alpha_key = _canonical_sym_pair(*alpha_key)

# Keep only one representative of each symmetric pair
unique_terms = {}
for (A, B), entry in avg_P1D_catalog.items():
    canon = _canonical_sym_pair(A, B)
    if canon not in unique_terms:
        unique_terms[canon] = entry

if alpha_key not in unique_terms:
    raise KeyError(f"Could not find the alpha auto term {alpha_key} in avg_P1D_catalog.")

entry_alpha = unique_terms[alpha_key]

# Robust key access
if "P1D_mean" in entry_alpha:
    P_alpha = np.real(np.asarray(entry_alpha["P1D_mean"]))
elif "p1d_mean" in entry_alpha:
    P_alpha = np.real(np.asarray(entry_alpha["p1d_mean"]))
else:
    raise KeyError("Could not find 'P1D_mean' or 'p1d_mean' in alpha entry.")

omega = np.asarray(omega, dtype=float)
P_total = np.real(np.asarray(P1D_tot_mean))
omega_tot = np.asarray(omega_tot, dtype=float)

# Interpolate alpha onto the total grid if needed
if len(omega) != len(omega_tot) or not np.allclose(omega, omega_tot):
    idx = np.argsort(omega)
    omega_alpha_sorted = omega[idx]
    P_alpha_sorted = P_alpha[idx]
    P_alpha_on_tot = np.interp(omega_tot, omega_alpha_sorted, P_alpha_sorted, left=np.nan, right=np.nan)
else:
    P_alpha_on_tot = P_alpha

# Compute quotient
ratio = P_total / P_alpha_on_tot

mask = (
    (omega_tot >= 0.0) & (omega_tot <= max_x) & np.isfinite(ratio) & np.isfinite(P_total) & np.isfinite(P_alpha_on_tot) & (P_alpha_on_tot != 0.0)
)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(omega_tot[mask], ratio[mask], lw=2.0, color="k")

ax.set_xlim(0.0, max_x)
ax.set_ylim(y_min, y_max)
ax.set_xscale("linear")
ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)

plt.tight_layout()
plt.show()

# $\xi1D$

In [ ]:
# Convert P1D -> Xi1D
Xi1D = mcfs.Xi1D.Xi1DAnalyzer(lambda_or_v=v_kms, optical_depth_cube=tau_cube)

lags_tot, Xi1D_tot = Xi1D.compute_total_Xi1D_from_P1D(
    P1D_total=P1D_tot, axis=-1, P1D_zero_mode_dropped=True, center_lags=False,
)
Xi1D_tot_mean = np.mean(Xi1D_tot, axis=0)

lags, Xi1D_catalog, by_order_xi = Xi1D.compute_subset_Xi1D_catalog_from_P1D(
    P1D_catalog=P1D_catalog, axis=-1, P1D_zero_mode_dropped=True,   # because drop_zero_mode=True in P1DAnalyzer
    center_lags=False,            # or True if you prefer centered signed lags
)

lags, avg_Xi1D_catalog = Xi1D.compute_average_Xi1D_catalog(
    Xi1D_catalog=Xi1D_catalog, average_axis=0, return_std=False, return_sem=True, symmetrize=True,
)

# Some checks
print(f"lags_tot shape: {lags_tot.shape}")
print(f"Xi1D_tot shape: {Xi1D_tot.shape}")
print(f"Xi1D_tot_mean shape: {Xi1D_tot_mean.shape}")

print(f"number of Xi1D catalog entries: {len(Xi1D_catalog)}")
print(f"available total orders in Xi1D catalog: {sorted(by_order_xi.keys())}")

example_key = next(iter(avg_Xi1D_catalog))
print(f"example averaged xi key: {example_key}")
print(f"example xi label: {avg_Xi1D_catalog[example_key]['label']}")
print(f"example xi mean shape: {avg_Xi1D_catalog[example_key]['Xi1D_mean'].shape}")
if 'Xi1D_sem' in avg_Xi1D_catalog[example_key]:
    print(f"example xi sem shape: {avg_Xi1D_catalog[example_key]['Xi1D_sem'].shape}")

In [ ]:
# Configuration
positive_lags_only = True
max_x = float(np.max(lags)) if positive_lags_only else None

yscale = "linear"      # change to "symlog" if desired
y_min = -0.015
y_max = 0.005
y_min = None
y_max = None
y_min = -0.1
y_max = 0.2

show_line_pair_markers = True

xlabel = r"$\Delta \left[\mathrm{km\,s^{-1}}\right]$"
ylabel = r"$\xi_{\mathrm{1D}}$"

# Group terms by symmetric pair, but keep both members
grouped_terms = {}
for (A, B), entry in avg_Xi1D_catalog.items():
    canon = _canonical_sym_pair(A, B)
    grouped_terms.setdefault(canon, []).append(((A, B), entry))

# Sort groups by total order, then lexicographically by canonical pair
sorted_groups = sorted(
    grouped_terms.items(),
    key=lambda item: (item[1][0][1]["total_order"], item[0][0], item[0][1])
)

orders = sorted({group[0][1]["total_order"] for _, group in sorted_groups})
order_to_ls = {order: linestyles[i % len(linestyles)] for i, order in enumerate(orders)}

# Prepare lag masks
lags = np.asarray(lags, dtype=float)
lags_tot = np.asarray(lags_tot, dtype=float)

if positive_lags_only:
    mask_terms = (lags >= 0.0)
    mask_total = (lags_tot >= 0.0)
else:
    mask_terms = np.ones_like(lags, dtype=bool)
    mask_total = np.ones_like(lags_tot, dtype=bool)

if max_x is not None:
    mask_terms &= (lags <= max_x)
    mask_total &= (lags_tot <= max_x)

fig, ax = plt.subplots(figsize=(12, 7))

term_handles = []
term_labels = []

Xi1D_sum_terms = np.zeros_like(lags[mask_terms])
for i, (canon, group) in enumerate(sorted_groups):
    color = colors[i % len(colors)]

    # Sort members so (A,B) and (B,A) appear together
    group = sorted(group, key=lambda x: (x[0][0], x[0][1]))

    for j, ((A, B), entry) in enumerate(group):

        print(i, j, A, B, entry["total_order"])

        mean = np.real(np.asarray(entry["Xi1D_mean"]))
        sem  = np.asarray(entry.get("Xi1D_sem", np.zeros_like(mean)))

        order = entry["total_order"]
        ls = order_to_ls[order]

        A_lab = _subset_to_text(A)
        B_lab = _subset_to_text(B)
        label = f"ξ[{A_lab}|{B_lab}]"

        Xi1D_sum_terms += mean[mask_terms]

        # Same color/style for both symmetric terms; slightly different alpha
        line_alpha = 0.95 if j == 0 else 0.65
        band_alpha = 0.15 if j == 0 else 0.08

        ax.plot(lags[mask_terms], mean[mask_terms], lw=2, ls=ls, color=color, alpha=line_alpha)
        ax.fill_between(
            lags[mask_terms], mean[mask_terms] - sem[mask_terms], mean[mask_terms] + sem[mask_terms],
            color=color, alpha=band_alpha,
        )

        term_handles.append(Line2D([0], [0], color=color, lw=2, ls=ls, alpha=line_alpha))
        term_labels.append(label)

# Total correlation
ax.plot(lags_tot[mask_total], np.real(np.asarray(Xi1D_tot_mean))[mask_total], lw=1.4, ls="-", color="k", label="total")

Xi1D_recovered = (Xi1D_sum_terms - C_means[-1]**2) / (1 + C_means[-1])**2
# ax.plot(lags[mask_terms], Xi1D_recovered, lw=2.0, ls="--", color="gray", label="sum Xi1D terms")

# ------------------------------------------------------------
# Vertical markers for line pairs
# ------------------------------------------------------------
if show_line_pair_markers and (line_bundle is not None):
    lag_max_for_markers = max_x if max_x is not None else float(np.max(lags[mask_terms]))
    vertical_markers = mcfs.plotting_utils._compute_line_pair_markers(
        line_bundle=line_bundle, lag_max=lag_max_for_markers, positive_lags_only=positive_lags_only,
    )

    mcfs.plotting_utils._draw_vertical_markers(
        ax, vertical_markers, xlim=(0.0, lag_max_for_markers) if positive_lags_only else None,
        color="k", lw=0.9, ls="--", alpha=0.30, text_fontsize=18, text_alpha=0.75, show_text=True,
    )

# ------------------------------------------------------------
# Axes styling
# ------------------------------------------------------------
if positive_lags_only:
    ax.set_xlim(0.0, max_x if max_x is not None else np.max(lags[mask_terms]))
elif max_x is not None:
    ax.set_xlim(np.min(lags[mask_terms]), max_x)

ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)
ax.set_yscale(yscale)

if (y_min is not None) or (y_max is not None):
    ax.set_ylim(y_min, y_max)

ax.axhline(0.0, color="k", lw=0.8, alpha=0.7)

# Legend for terms
leg1 = ax.legend(term_handles, term_labels, loc="upper right", fontsize=14, ncol=2, frameon=True)
ax.add_artist(leg1)

# Legend for perturbative / total order
order_handles = [
    Line2D([0], [0], color="k", lw=2, ls=order_to_ls[o], label=rf"$\mathcal{{O}}({o})$")
    for o in orders
]
order_handles.append(Line2D([0], [0], color="k", lw=1.4, ls="-", label="total"))

ax.legend(handles=order_handles, loc="lower left", fontsize=14, frameon=True, ncol=len(order_handles))

plt.tight_layout()
plt.show()